In [9]:
import pandas as pd

base = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
fs = pd.read_csv(base + "finess_sygen_final.csv", dtype=str)
insee = pd.read_csv(base + "insee_communes_final.csv", dtype=str)

In [10]:
def harmoniser(code):
    if pd.isna(code):
        return code
    if "75101" <= code <= "75120":
        return "75056"   # Paris
    if "69381" <= code <= "69389":
        return "69123"   # Lyon
    if "13201" <= code <= "13216":
        return "13055"   # Marseille
    return code

fs2 = fs.dropna(subset=["CODGEO"]).copy()
fs2["CODGEO_ORIG"] = fs2["CODGEO"]
fs2["CODGEO"] = fs2["CODGEO"].map(harmoniser)
fs3 = fs2[fs2["ETAT"] == "A"].copy()

print(len(fs), len(fs2), len(fs3))

3988 3940 3921


In [11]:
cols_num = ["LIT_MED", "LIT_CHI", "LIT_OBS", "LIT_MCO", "LIT_SMR", "LIT_USLD",
            "NB_SCAN", "NB_IRM", "NB_MAMM"]
for c in cols_num:
    fs3[c] = pd.to_numeric(fs3[c], errors="coerce")

hopitaux = fs3.groupby("CODGEO")[cols_num].sum(min_count=1)
hopitaux["nb_etablissements"] = fs3.groupby("CODGEO")["FI"].count()
hopitaux = hopitaux.reset_index()

print(hopitaux.shape)
print(hopitaux["nb_etablissements"].sum())
print(hopitaux.head())

(1714, 11)
3921
  CODGEO  LIT_MED  LIT_CHI  LIT_OBS  LIT_MCO  LIT_SMR  LIT_USLD  NB_SCAN  \
0  01004     31.0     27.0     15.0     73.0      NaN       NaN      NaN   
1  01033      NaN      NaN      NaN      NaN      NaN       NaN      NaN   
2  01034     56.0     19.0      8.0     83.0     40.0       NaN      1.0   
3  01053     26.0     44.0      NaN     70.0    156.0      11.0      0.0   
4  01082      NaN      NaN      NaN      NaN     70.0       NaN      NaN   

   NB_IRM  NB_MAMM  nb_etablissements  
0     NaN      NaN                  3  
1     NaN      NaN                  1  
2     0.0      1.0                  2  
3     0.0      0.0                  5  
4     NaN      NaN                  1  
